<a href="https://colab.research.google.com/github/likhitnayak17-dev/AI_lab/blob/main/lab6_1WN24CS146_LIKHIT_MAHESH_NAYAK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from itertools import product
import re

# Supported operations: AND, OR, NOT, IMPLIES, IFF

def tokenize(expr):
    pattern = r'\(|\)|[A-Za-z][A-Za-z0-9_]*'
    tokens = re.findall(pattern, expr.upper())

    if re.sub(pattern, '', expr.upper()).strip():
        raise ValueError("Invalid characters in expression.")

    return tokens


def to_rpn(tokens):
    precedence = {
        'IFF': 1,
        'IMPLIES': 2,
        'OR': 3,
        'AND': 4,
        'NOT': 5
    }

    right_assoc = {'NOT', 'IMPLIES', 'IFF'}
    output = []
    operators = []

    for token in tokens:
        if token in ('P', 'Q', 'R'):
            output.append(token)

        elif token == '(':
            operators.append(token)

        elif token == ')':
            while operators and operators[-1] != '(':
                output.append(operators.pop())

            if not operators:
                raise ValueError("Mismatched parentheses.")
            operators.pop()

        elif token in precedence:
            while operators and operators[-1] in precedence:
                top = operators[-1]

                if ((token not in right_assoc
                     and precedence[token] <= precedence[top])
                    or (token in right_assoc
                        and precedence[token] < precedence[top])):
                    output.append(operators.pop())
                else:
                    break

            operators.append(token)

        else:
            raise ValueError(
                f"Invalid token: {token}. Use P, Q, R and operators."
            )

    while operators:
        op = operators.pop()
        if op == '(':
            raise ValueError("Mismatched parentheses.")
        output.append(op)

    return output


def evaluate(rpn, model):
    stack = []

    for token in rpn:
        if token in ('P', 'Q', 'R'):
            stack.append(model[token])

        elif token == 'NOT':
            if len(stack) < 1:
                raise ValueError("Invalid NOT expression.")
            stack.append(not stack.pop())

        else:
            if len(stack) < 2:
                raise ValueError("Invalid expression.")

            b = stack.pop()
            a = stack.pop()

            if token == 'AND':
                result = a and b
            elif token == 'OR':
                result = a or b
            elif token == 'IMPLIES':
                result = (not a) or b
            elif token == 'IFF':
                result = a == b

            stack.append(result)

    if len(stack) != 1:
        raise ValueError("Invalid expression.")

    return stack[0]


def main():
    print("PROPOSITIONAL LOGIC - TRUTH TABLE ENUMERATION")
    print("Variables: P, Q, R")
    print("Operators: AND, OR, NOT, IMPLIES, IFF\n")

    kb1 = input("Enter KB1: ")
    kb2 = input("Enter KB2: ")
    kb3 = input("Enter KB3: ")
    query = input("Enter Query: ")

    try:
        kb_rpns = [
            to_rpn(tokenize(kb1)),
            to_rpn(tokenize(kb2)),
            to_rpn(tokenize(kb3))
        ]
        query_rpn = to_rpn(tokenize(query))

        if any(not expr for expr in kb_rpns) or not query_rpn:
            raise ValueError("All inputs must be non-empty.")

        all_exprs = kb_rpns + [query_rpn]
        if any(
            not any(t in ('P', 'Q', 'R') for t in expr)
            for expr in all_exprs
        ):
            raise ValueError("Each expression must contain P, Q, or R.")

        print("\nTruth Table")
        print("P\tQ\tR\tKB1\tKB2\tKB3\tCombined KB\tQuery")
        print("-" * 65)

        entails = True
        counterexample = None

        for values in product([True, False], repeat=3):
            model = dict(zip(('P', 'Q', 'R'), values))

            v1 = evaluate(kb_rpns[0], model)
            v2 = evaluate(kb_rpns[1], model)
            v3 = evaluate(kb_rpns[2], model)

            # All three KB statements must be true
            combined_kb = v1 and v2 and v3
            q_value = evaluate(query_rpn, model)

            row = [
                model['P'], model['Q'], model['R'],
                v1, v2, v3, combined_kb, q_value
            ]

            print("\t".join(
                'T' if value else 'F' for value in row
            ))

            if combined_kb and not q_value:
                entails = False
                counterexample = model.copy()

        print("\nRESULT:")
        if entails:
            print("KB ENTAILS the Query (KB |= Query)")
        else:
            print("KB DOES NOT ENTAIL the Query (KB |/= Query)")
            print("Counterexample:", counterexample)

    except ValueError as error:
        print("Input Error:", error)


if __name__ == "__main__":
    main()


PROPOSITIONAL LOGIC - TRUTH TABLE ENUMERATION
Variables: P, Q, R
Operators: AND, OR, NOT, IMPLIES, IFF

Enter KB1: P IFF Q
Enter KB2: Q AND R
Enter KB3: NOT P IMPLIES R
Enter Query: Q

Truth Table
P	Q	R	KB1	KB2	KB3	Combined KB	Query
-----------------------------------------------------------------
T	T	T	T	T	T	T	T
T	T	F	T	F	T	F	T
T	F	T	F	F	T	F	F
T	F	F	F	F	T	F	F
F	T	T	F	T	T	F	T
F	T	F	F	F	F	F	T
F	F	T	T	F	T	F	F
F	F	F	T	F	F	F	F

RESULT:
KB ENTAILS the Query (KB |= Query)
